### Database creation for piano log project

In [1]:
import sqlite3

conn = sqlite3.connect("piano.db")
cursor = conn.cursor()

# --- COMPOSERS ---
cursor.execute("""
CREATE TABLE IF NOT EXISTS composers (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    normalized_name TEXT UNIQUE
)
""")

# --- PIECES ---
cursor.execute("""
CREATE TABLE IF NOT EXISTS pieces (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    canonical_name TEXT NOT NULL,
    normalized_name TEXT,
    composer_id INTEGER,
    opus TEXT,
    number TEXT,
    key TEXT,

    UNIQUE(canonical_name, composer_id),

    FOREIGN KEY (composer_id) REFERENCES composers(id)
)
""")

# --- ALIASES ---
cursor.execute("""
CREATE TABLE IF NOT EXISTS piece_aliases (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    alias TEXT NOT NULL,
    normalized_alias TEXT,
    piece_id INTEGER,

    FOREIGN KEY (piece_id) REFERENCES pieces(id)
)
""")

# --- SESSIONS ---
cursor.execute("""
CREATE TABLE IF NOT EXISTS sessions (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    date TEXT NOT NULL,
    duration_min INTEGER,
    raw_text TEXT
)
""")

# --- ACTIVITIES ---
cursor.execute("""
CREATE TABLE IF NOT EXISTS activities (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    session_id INTEGER,

    type TEXT,

    piece_id INTEGER,
    composer_id INTEGER,

    piece_name TEXT,     -- raw text (important!)
    composer_name TEXT,  -- raw text

    key TEXT,
    section TEXT,
    bars TEXT,
    focus TEXT,
    notes TEXT,

    FOREIGN KEY (session_id) REFERENCES sessions(id),
    FOREIGN KEY (piece_id) REFERENCES pieces(id),
    FOREIGN KEY (composer_id) REFERENCES composers(id)
)
""")

conn.commit()

In [2]:
def normalize(text):
    return text.lower().strip()

In [3]:
def get_or_create_composer(name):
    if not name:
        return None

    norm = normalize(name)

    cursor.execute(
        "SELECT id FROM composers WHERE normalized_name = ?",
        (norm,)
    )
    row = cursor.fetchone()

    if row:
        return row[0]

    cursor.execute(
        "INSERT INTO composers (name, normalized_name) VALUES (?, ?)",
        (name, norm)
    )
    return cursor.lastrowid

In [4]:
def get_or_create_piece(piece_name, composer_id):
    if not piece_name:
        return None

    norm = normalize(piece_name)

    cursor.execute("""
        SELECT id FROM pieces
        WHERE normalized_name = ? AND composer_id IS ?
    """, (norm, composer_id))

    row = cursor.fetchone()
    if row:
        return row[0]

    cursor.execute("""
        INSERT INTO pieces (canonical_name, normalized_name, composer_id)
        VALUES (?, ?, ?)
    """, (piece_name, norm, composer_id))

    return cursor.lastrowid

In [5]:
def log_session(session, raw_text):
    cursor.execute(
        "INSERT INTO sessions (date, duration_min, raw_text) VALUES (?, ?, ?)",
        (session["date"], session["duration_min"], raw_text)
    )
    session_id = cursor.lastrowid

    for act in session["activities"]:
        composer_id = get_or_create_composer(act.get("composer"))
        piece_id = get_or_create_piece(act.get("piece_name"), composer_id)

        cursor.execute("""
            INSERT INTO activities (
                session_id, type,
                piece_id, composer_id,
                piece_name, composer_name,
                key, section, bars, focus, notes
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            session_id,
            act.get("type"),
            piece_id,
            composer_id,
            act.get("piece_name"),
            act.get("composer"),
            act.get("key"),
            act.get("section"),
            act.get("bars"),
            act.get("focus"),
            act.get("notes"),
        ))

    conn.commit()

In [6]:
cursor.execute("""
SELECT p.canonical_name, COUNT(*)
FROM activities a
JOIN pieces p ON a.piece_id = p.id
GROUP BY p.canonical_name
""")

print(cursor.fetchall())

[]
